# 154 — Deriva, feedback y evaluación continua

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Términos:
Q1: (0.30−0.25)·ln(1.20) = 0.05·0.182 = **0.0091**
Q2: (0.28−0.25)·ln(1.12) = 0.03·0.113 = **0.0034**
Q3: (0.24−0.25)·ln(0.96) = (−0.01)·(−0.041) = **0.0004**
Q4: (0.18−0.25)·ln(0.72) = (−0.07)·(−0.329) = **0.0230**
PSI ≈ **0.036 < 0.10 → estable**. La población se movió levemente hacia los bins bajos
(más jóvenes), con la mayor contribución en Q4 (déficit de mayores). Nótese que un
desplazamiento visible a ojo puede seguir siendo «estable» según PSI: el umbral 0.10 es
deliberadamente poco sensible.

**Ejercicio 2.** a) **Deriva de datos** (`P(X)` cambia: país nuevo); el PSI de la feature
`pais` la detecta de inmediato (aparece masa en una categoría nueva). b) **Concepto**
(`P(Y|X)` cambia con perfil idéntico); el PSI de features NO la ve — la detectan las
etiquetas diferidas o, antes, la deriva de la tasa de impago observada. c) **Deriva de
etiquetas** por redefinición — en rigor, un cambio de contrato del problema: `P(Y)`
cambia porque `Y` ya es otra variable. Ningún monitor de X la ve; solo la gobernanza del
esquema de etiquetas (y la caída aparente de métricas) la revela.

**Ejercicio 3.** `q_3/p_3 = 0.2/0` es división por cero (y `p_3 ln(p_3)` no aparece,
pero el término exige `ln(q/p)`). Con ε = 0.001: término ≈ (0.2−0.001)·ln(0.2/0.001) =
0.199·ln(200) ≈ 0.199·5.298 ≈ **1.054** → PSI total ≳ 1, gigantesco. Lección: la
aparición de masa en un bin que el baseline consideraba imposible dispara el PSI
violentamente — deseable como alarma (es un evento cualitativamente nuevo), pero
sensible al ε elegido, por lo que el valor numérico exacto no es interpretable, solo la
señal.

**Ejercicio 4.** Política ejemplo: **seguir** si PSI < 0.10 (registro semanal);
**investigar** si 0.10–0.25 — exige localizar bins responsables, revisar cambios de
ingesta y comparar el desempeño del segmento afectado con etiquetas disponibles;
**actuar** si > 0.25 — exige confirmación con una segunda señal (deriva del score o
caída de métrica en cohorte madura) antes de reentrenar, y corrección de datos si la
causa es un bug. El laboratorio aporta la evidencia estructurada sobre la que esta
política operaría.


In [ ]:
result = run_lab("evaluation", seed=154)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
import math

def psi(p, q, eps=1e-3):
    total = 0.0
    for pi, qi in zip(p, q):
        pi = max(pi, eps)
        qi = max(qi, eps)
        total += (qi - pi) * math.log(qi / pi)
    return total

p = [0.25, 0.25, 0.25, 0.25]
q = [0.30, 0.28, 0.24, 0.18]
print(f"Ejercicio 1: PSI = {psi(p, q):.4f}  (estable: < 0.10)")

p3 = [0.5, 0.5, 0.0]
q3 = [0.4, 0.4, 0.2]
print(f"Ejercicio 3: PSI con suavizado = {psi(p3, q3):.3f}  (bin nuevo dispara la señal)")


## Reflexión

1. Construye un ejemplo concreto de concept drift donde `P(X)` no cambie en absoluto: ¿qué señal lo detectaría y con cuánto retardo?
2. ¿Por qué cada término del PSI es no negativo, y qué implica eso sobre la posibilidad de que dos cambios «se cancelen» entre bins?
3. Tu PSI del score dio 0.30 pero la métrica con etiquetas diferidas (cohorte de hace 90 días) sigue estable: ¿qué tres hipótesis explican la discrepancia y cómo distinguirías entre ellas?
